# 01 – Question

**Projekt:** WealthScope AI 1.0
**Methode:** QUA³CK · reproduzierbarer Out-of-Time-Benchmark
**Hinweis:** Wissenschaftlicher Prototyp, keine Anlageberatung.

## Leitfrage

**Wie können historische US-Aktienmarktdaten genutzt werden, um eine interaktive
Finanzanalyse-App zu entwickeln, die technische Analyse, ML-Signalgebung und
risikobasierte Positionsplanung nachvollziehbar kombiniert?**

### Hypothesen

- **H1:** Der Random Forest erreicht im purged Out-of-Time-Test eine ROC-AUC
  oberhalb von 0,5 und bleibt im Walk-forward-Vergleich stabil.
- **H2:** Die Streamlit-App macht Daten, Methodik und Grenzen für
  Nicht-Experten zugänglich.
- **H3:** Die Kombination aus technischen Indikatoren, ML und Risiko verbessert
  die Orientierung; ein wirtschaftlicher Nutzen muss separat getestet werden.

## Lernziele

Nach diesem Notebook könnt ihr:

- eine Hypothese so formulieren, dass sie scheitern kann
- das Prognoseziel vom Prognosezeitpunkt her denken
- Erfolgskriterien vor der ersten Modellzeile festlegen

## Ausgangslage

Privatanleger stehen vor Datenfülle, widersprüchlichen Signalen und
Werkzeugen, die ihre eigene Treffsicherheit nicht offenlegen. Die naheliegende
Projektidee — „ein ML-Modell, das Kurse vorhersagt" — ist deshalb weniger
interessant als die Frage dahinter: **Wie viel Signal steckt überhaupt in reinen
Kursdaten, wenn man ehrlich misst?**

Diese Umformulierung ist der eigentliche Projektentscheid. Sie macht ein
schwaches Ergebnis zu einem Befund statt zu einem Misserfolg.

## Falsifikationskriterien

Eine Hypothese, die jeder Ausgang bestätigt, ist wertlos. Deshalb steht
**vor** dem Training fest, was sie widerlegen würde:

| Hypothese | Bestätigt, wenn … | Widerlegt, wenn … | Geprüft in |
|---|---|---|---|
| **H1** | ROC-AUC deutlich über 0,5 **und** über die Walk-forward-Folds stabil | AUC nahe 0,5 oder über Zeitfenster instabil | 04, 05 |
| **H2** | Methodik, Kennzahlen und Grenzen sind in der App ohne Vorwissen auffindbar | zentrale Kennzahlen fehlen oder sind unerklärt | 06 |
| **H3** | messbarer Zusatznutzen gegenüber Einzelindikatoren | kein Nutzen nachweisbar oder nicht prüfbar | 05 |

> **Wichtig für H1:** „AUC über 0,5" allein genügt nicht. Bei 69.147 Testfällen
> wird auch ein winziger Vorsprung statistisch signifikant. Notebook 05 prüft
> deshalb zusätzlich per Bootstrap, wie groß der Vorsprung wirklich ist — und
> trennt statistische Signifikanz von praktischer Bedeutung.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wealthscope_features.parquet"
DIAGNOSTICS_PATH = PROJECT_ROOT / "models" / "diagnostics.json"
EXPERIMENTS_PATH = PROJECT_ROOT / "models" / "validation_experiments.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datensatz fehlt: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Daten: {len(df):,} Zeilen × {len(df.columns)} Spalten")
display(df.head(3))

Daten: 192,119 Zeilen × 27 Spalten


,date,open,high,low,close,volume,ticker,asset_type,source_file,daily_return,...,ma_200_distance,volatility_20d,rolling_high,drawdown,future_return_20d,target_20d,open_interest,volatility_60d,rolling_low_60,rolling_high_60
0,1985-06-21,0.25742,0.26381,0.25742,0.25742,46333854,AAPL,Stock,Stocks/aapl.us.txt,0.020374,...,-0.331288,0.040521,0.48791,-0.472403,0.044635,1.0,NaN,NaN,NaN,NaN
1,1985-06-24,0.27535,0.27920,0.27535,0.27535,57384755,AAPL,Stock,Stocks/aapl.us.txt,0.069653,...,-0.283328,0.040440,0.48791,-0.435654,-0.041910,0.0,NaN,NaN,NaN,NaN
2,1985-06-25,0.27920,0.28556,0.27920,0.27920,81966629,AAPL,Stock,Stocks/aapl.us.txt,0.013982,...,-0.271961,0.037120,0.48791,-0.427763,-0.073424,0.0,NaN,NaN,NaN,NaN


In [2]:
questions = pd.DataFrame([
    ["Q1", "Welche Daten liegen wirklich vor?", "Zeilen, Ticker, Zeitraum, Fehlwerte"],
    ["Q2", "Was ist das Prognoseziel?", "target_20d: Richtung nach 20 Handelstagen"],
    ["Q3", "Wie wird Leakage verhindert?", "Zeit-Split, 20T-Purge, Pipeline"],
    ["Q4", "Welches Modell ist tragfähig?", "Dummy bis Random Forest, gleiche Fenster"],
    ["Q5", "Wie wird Wissen übertragen?", "App, Model Card, Export, Lernstudio"],
], columns=["ID", "Frage", "Operationalisierung"])
questions

,ID,Frage,Operationalisierung
0,Q1,Welche Daten liegen wirklich vor?,"Zeilen, Ticker, Zeitraum, Fehlwerte"
1,Q2,Was ist das Prognoseziel?,target_20d: Richtung nach 20 Handelstagen
2,Q3,Wie wird Leakage verhindert?,"Zeit-Split, 20T-Purge, Pipeline"
3,Q4,Welches Modell ist tragfähig?,"Dummy bis Random Forest, gleiche Fenster"
4,Q5,Wie wird Wissen übertragen?,"App, Model Card, Export, Lernstudio"


In [3]:
scope = pd.DataFrame([
    ["Im Scope", "Klassifikation, technische Features, Erklärbarkeit, Risikoszenarien"],
    ["Nicht im Scope", "Handelsbot, sichere Prognosen, persönliche Anlageberatung"],
    ["Erfolgskriterium", "Reproduzierbare und kritisch interpretierte Ergebnisse"],
], columns=["Bereich", "Festlegung"])
scope

,Bereich,Festlegung
0,Im Scope,"Klassifikation, technische Features, Erklärbar..."
1,Nicht im Scope,"Handelsbot, sichere Prognosen, persönliche Anl..."
2,Erfolgskriterium,Reproduzierbare und kritisch interpretierte Er...


## Management-Checkpoint

Das Erfolgskriterium dieses Projekts ist **nicht** eine hohe Kennzahl, sondern
eine Kennzahl, der man glauben kann. Wer H1 vor dem Training so formuliert, dass
sie scheitern kann, darf ein schwaches Ergebnis später als Befund berichten
statt es zu verstecken. Umgekehrt gilt: Wäre H1 als „das Modell findet Muster"
formuliert worden, hätte jedes Ergebnis sie bestätigt — und die Arbeit wäre
wissenschaftlich wertlos.